# 01 · Vocabulary and Tokens
**Goal:** understand how raw text becomes the integers a Transformer can consume.
Concepts: *token*, *vocabulary*, *token id*, *special tokens* (`<PAD>`, `<SOS>`, `<EOS>`, `<UNK>`), *vocab size*, *subword tokenization*.
This notebook is standalone — it only needs `numpy`. Run the cells top to bottom.

## Words you'll meet
- **Token**: one piece of text the model reads, like one Lego brick in a model. It might be a whole word, a letter, or a chunk of a word.
- **Vocabulary**: the list of every token the model knows, like the catalogue of a library.
- **Token id**: the number given to a token, like a library shelf number or a number in the class register.
- **Special tokens**: reserved "signal" tokens such as start, end and unknown, like the bell that tells a lesson to begin or finish.
- **Vocab size (V)**: how many entries the vocabulary has, i.e. how many names are in the register.
- **Subword / BPE**: spelling words with a few reusable bricks (like `low` + `er`) instead of one special brick for every word.
- **Embedding table**: a big table with one row of numbers per token, like a timetable with one row per class.

## 1. A neural network only sees numbers
A model can't multiply the word "cat". Step one of every language model is:
`text → tokens → integer ids`
A **token** is the unit the model reads. It could be a word, a character, or a piece of a word.

In [ ]:
import numpy as np
from collections import Counter
text = "the cat sat on the mat . the dog sat on the log ."
tokens = text.split()          # simplest tokenizer: split on spaces
print(tokens)
print("number of tokens:", len(tokens))

> 🧒 **In plain words:**
> `text.split()` cuts the sentence into pieces wherever there is a space, giving a list of words. We call each word a token.
> `len(tokens)` counts the list. **Notice:** 14 tokens, and the full stop `.` counts as a token too. The word `the` appears 4 times, so tokens repeat.
> **Try it:** change the sentence to anything you like and re-run; the count should match the number of words you typed.

## 2. Vocabulary = the model's dictionary
The **vocabulary** is the set of all distinct tokens the model knows. Each gets a unique integer **id**.
`vocab_size` (often written `V`) later decides the size of the embedding table and of the output layer.

In [ ]:
counts = Counter(tokens)
print(counts)
unique_tokens = sorted(counts)           # sorted for reproducibility
print("\nunique tokens:", unique_tokens)
print("vocab size (without specials):", len(unique_tokens))

> 🧒 **In plain words:**
> `Counter(tokens)` is a tally sheet: it counts how many times each word appears. `sorted(counts)` lists the different words in alphabetical order, with each word once.
> **Notice:** there are 14 tokens but only 8 different ones, so the vocabulary is 8. `the` has the highest tally (4).
> Think of it like a library catalogue: it lists each book once, even if the library owns many copies.

## 3. Special tokens
Real vocabularies reserve ids for control tokens:
| token | meaning |
|---|---|
| `<PAD>` | filler so sequences in a batch have equal length (notebook 02) |
| `<SOS>` | start of sequence — the decoder's first input |
| `<EOS>` | end of sequence — "I'm done generating" |
| `<UNK>` | unknown — any word not in the vocabulary |
We put them first so their ids are stable.

In [ ]:
specials = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
itos = specials + unique_tokens                  # id -> token
stoi = {t: i for i, t in enumerate(itos)}        # token -> id
print("vocab size V =", len(itos))
for i, t in enumerate(itos):
    print(f"{i:2d}  {t}")

> 🧒 **In plain words:**
> `itos` ("id to string") is a list: put the 4 special tokens first, then our 8 words. A token's position in the list is its id number.
> `stoi` ("string to id") is the reverse: a dictionary where you look up a word and get its number. The line with `enumerate` goes through the list and pairs each word with its position.
> **Notice:** `<PAD>` is 0, `<SOS>` is 1, and `the` is 11. The total V is now 12 (8 words + 4 specials).
> Think of it like a class register: every name gets a number.

## 4. Encode and decode
- **encode**: text → list of ids (unknown words become `<UNK>`)
- **decode**: ids → text

In [ ]:
def encode(sentence, add_sos=False, add_eos=False):
    ids = [stoi.get(w, stoi["<UNK>"]) for w in sentence.split()]
    if add_sos: ids = [stoi["<SOS>"]] + ids
    if add_eos: ids = ids + [stoi["<EOS>"]]
    return ids
def decode(ids):
    return " ".join(itos[i] for i in ids)
ids = encode("the cat sat on the mat", add_sos=True, add_eos=True)
print("ids    :", ids)
print("decoded:", decode(ids))

> 🧒 **In plain words:**
> `encode` splits a sentence into words and looks up each one's id. `stoi.get(w, stoi["<UNK>"])` means "find the id of this word, and if it is not in the dictionary, use the id of `<UNK>` instead". The two `if` lines glue a start marker and/or end marker onto the list. `decode` does the opposite: it turns each id back into a word and joins them with spaces.
> **Notice:** the ids start with 1 (start) and end with 2 (end), and decoding gives back the original sentence wrapped in `<SOS>` and `<EOS>`.
> **Try it:** encode `"the dog sat on the log"`; you should see ids 11, 6, 10, 9, 11, 7 between the markers.

### Try it: an unseen word
"The elephant sat" — "elephant" was never in our corpus.

In [ ]:
ids = encode("the elephant sat")
print(ids, "->", decode(ids))
print("\nThe information 'elephant' is lost: it became <UNK>.")

> 🧒 **In plain words:**
> We give the encoder a word it has never seen. The dictionary has no entry, so it falls back to `<UNK>` (id 3).
> **Notice:** `[11, 3, 10]` decodes to `the <UNK> sat`. The word "elephant" is gone for good, like replacing a name with "someone" and then not being able to recover it.

## 5. The out-of-vocabulary problem and subwords
Word-level vocabularies explode (millions of words, plus typos, names, new words) and still produce `<UNK>`.
| granularity | vocab size | sequence length | issue |
|---|---|---|---|
| character | tiny (~100) | very long | model must learn spelling → meaning |
| word | huge | short | unknown words |
| **subword (BPE)** | moderate (30k–100k) | medium | best trade-off; used by GPT, BERT, etc. |
Let's compare lengths for one sentence:

In [ ]:
sentence = "the cat sat on the mat"
print("word level :", len(sentence.split()), "tokens")
print("char level :", len(list(sentence)), "tokens")
print(list(sentence))

> 🧒 **In plain words:**
> We count the same sentence two ways. `sentence.split()` makes words; `list(sentence)` makes a list of single characters (spaces included).
> **Notice:** 6 word tokens versus 22 character tokens. Letters give a tiny vocabulary but very long sequences.
> **Try it:** swap in a longer sentence and see how much faster the character count grows.

## 6. Mini BPE (Byte-Pair Encoding)
BPE starts from characters and repeatedly **merges the most frequent adjacent pair** into a new token.
Frequent words become single tokens; rare words are split into reusable pieces. (Classic example from the BPE paper.)

In [ ]:
corpus = ("low " * 5 + "lower " * 2 + "newest " * 6 + "widest " * 3).split()
seqs = [list(w) + ["</w>"] for w in corpus]       # </w> marks end of word
def merge_pair(seq, a, b):
    out, i = [], 0
    while i < len(seq):
        if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(seq[i]); i += 1
    return out
for step in range(8):
    pairs = Counter()
    for s in seqs:
        for a, b in zip(s, s[1:]):
            pairs[(a, b)] += 1
    (a, b), n = pairs.most_common(1)[0]
    seqs = [merge_pair(s, a, b) for s in seqs]
    print(f"merge {step+1}: {a!r} + {b!r} -> {a+b!r}   (seen {n}x)")
print("\nfinal segmentation of each distinct word:")
for w in dict.fromkeys(corpus):
    print(f"  {w:8s}", seqs[corpus.index(w)])

> 🧒 **In plain words:**
> Start: every word is cut into letters plus an end-of-word marker `</w>`. Then the loop does 8 rounds. Each round it counts every neighbouring pair of pieces (`zip(s, s[1:])` pairs each piece with the one after it), finds the most common pair with `most_common(1)`, and glues that pair into one new piece everywhere using `merge_pair`.
> **Notice:** the first merges are `e`+`s`, then `es`+`t`, then `est`+`</w>`. These letters sit together in "newest" and "widest" a lot (9 times). Later, `low` and `newest</w>` become single pieces.
> Think of it like making a shortcut: if you type the same pair of letters all the time, you invent one key for it.
> At the end, `lower` is `low` + `e` + `r`, built from reusable bricks, while `widest` is `w`, `i`, `d` and `est</w>`.

Notice `newest` collapsed into a few big pieces while rarer words stay in smaller pieces. A *new* word like "lowest" could be built from `low` + `est</w>` — no `<UNK>` needed.

## 7. Why vocab size matters later
Two places in a Transformer depend on `V`:
1. **Input embedding table**: shape `(V, d_model)` — one row per token id (notebook 03).
2. **Output layer**: produces `V` scores (logits) for "which token comes next?" (notebook 07).
Bigger `V` = more parameters and a bigger softmax, but shorter sequences. In this repo's sort task the vocabulary is tiny: digits `0–9` plus `SOS` (id 11) and `EOS` (id 12).

In [ ]:
d_model = 512
for V in [13, 1_000, 32_000, 100_000]:
    print(f"V={V:>7,}  embedding params = {V*d_model:>12,}")

> 🧒 **In plain words:**
> Here we work out how big the embedding table is: V rows (one per token) times 512 numbers per row. The `_` in `1_000` is just a thousands separator for reading, and `:>12,` right-aligns the number with commas.
> **Notice:** with V=13 it is 6,656 numbers; with V=100,000 it is 51,200,000. The table grows in step with the vocabulary.
> **Try it:** change `d_model` to 256 and check the numbers halve.

## Recap
- **Token**: the unit of input. **Vocabulary**: the token ↔ id mapping. **`V`**: its size.
- **Special tokens** handle padding, start, end, unknown.
- **Subword** tokenization balances vocab size and sequence length.
➡ Next: **02 · Batch and Sequence** — how many token ids are stacked into a tensor.